# Análise do Espectro de Frequência em Imagens Sintéticas

**Disciplina:** Processamento Digital de Imagens  
**Autores:**
- Geovane Araujo de Lima Silva — RA: 00111884
- João Vitor Marinonio de Almeida

---

## Objetivo

Este notebook reproduz os experimentos centrais do artigo:

> Durall, R., Keuper, M., & Keuper, J. (2020). **Watch your Up-Convolution: CNN Based Generative Deep Neural Networks are Failing to Reproduce Spectral Distributions**. *CVPR 2020*. https://arxiv.org/abs/2003.01826

O método analisa a distribuição espectral de imagens reais e de imagens geradas por GANs usando a **Transformada de Fourier 2D (FFT)** e **integração azimutal** do espectro de potência.

### Hipótese do Artigo

Redes geradoras que utilizam operações de **up-convolution** (convolução transposta) introduzem distorções sistemáticas no espectro de frequência das imagens geradas. Essas distorções são perceptíveis no domínio da frequência mesmo quando invisíveis no domínio espacial — o que permite **detectar deepfakes automaticamente**.

---

## 1. Configuração e Importações

In [ ]:
import sys
import os

# Adiciona o diretório src ao path
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

from radial_profile import azimuthal_average, compute_power_spectrum_1d, rgb_to_gray
from spectral_utils import (
    load_image,
    load_images_from_dir,
    compute_mean_spectrum,
    plot_spectrum_comparison,
    plot_fft_visualization
)

# Configuração visual
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Bibliotecas carregadas com sucesso!')
print(f'NumPy:      {np.__version__}')
print(f'OpenCV:     {cv2.__version__}')
print(f'Matplotlib: {plt.matplotlib.__version__}')

---
## 2. Revisão Teórica: FFT e Espectro de Potência

### 2.1 Transformada de Fourier 2D

Para uma imagem $f(x,y)$ de dimensão $M \times N$, a DFT 2D é definida como:

$$F(u,v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y) \cdot e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}$$

O **espectro de potência** é dado pela magnitude ao quadrado:

$$P(u,v) = |F(u,v)|^2$$

Usamos a escala logarítmica em dB para melhor visualização:

$$P_{dB}(u,v) = 20 \cdot \log\left(|F(u,v)|\right)$$

### 2.2 Integração Azimutal

A integração azimutal reduz o espectro 2D a um perfil radial 1D, calculando a média dos coeficientes em anéis concêntricos de raio $r$:

$$\hat{P}(r) = \frac{1}{N_r} \sum_{(u,v): \lfloor\sqrt{u^2+v^2}\rfloor = r} P_{dB}(u,v)$$

Isso torna a análise **invariante à rotação** e permite comparar distribuições espectrais de diferentes imagens.

---
## 3. Demonstração com Imagens de Teste

Vamos gerar imagens sintéticas controladas para demonstrar o pipeline de análise espectral antes de aplicar em imagens reais.

In [ ]:
def generate_test_images():
    """
    Gera imagens sintéticas para demonstração:
    - Imagem com padrão de grade (alta frequência)
    - Imagem com gradiente suave (baixa frequência)
    - Imagem com ruído branco (espectro plano)
    - Imagem com padrão senoidal (frequência única)
    """
    size = 128
    x = np.linspace(0, 1, size)
    y = np.linspace(0, 1, size)
    xx, yy = np.meshgrid(x, y)

    # Gradiente suave (baixa frequência)
    gradiente = xx + yy
    gradiente = (gradiente - gradiente.min()) / gradiente.max()

    # Padrão senoidal (frequência controlada)
    freq = 8
    senoidal = 0.5 + 0.5 * np.sin(2 * np.pi * freq * xx)

    # Ruído branco
    np.random.seed(42)
    ruido = np.random.rand(size, size)

    # Padrão de grade (simula artefato de up-convolution)
    grade = np.zeros((size, size))
    grade[::4, :] = 1.0
    grade[:, ::4] = 1.0

    return {
        'Gradiente Suave\n(Baixa Frequência)': gradiente,
        'Padrão Senoidal\n(Frequência Única)': senoidal,
        'Ruído Branco\n(Espectro Plano)': ruido,
        'Padrão de Grade\n(Artefato Up-Conv)': grade,
    }

test_images = generate_test_images()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, (nome, img) in enumerate(test_images.items()):
    # Linha 0: imagem espacial
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(nome, fontsize=10)
    axes[0, col].axis('off')

    # Linha 1: espectro FFT
    fft = np.fft.fft2(img)
    fshift = np.fft.fftshift(fft)
    magnitude = 20 * np.log(np.abs(fshift) + 1e-8)
    axes[1, col].imshow(magnitude, cmap='inferno')
    axes[1, col].set_title('Espectro FFT (dB)', fontsize=10)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Domínio Espacial', fontsize=11, labelpad=10)
axes[1, 0].set_ylabel('Domínio da Frequência', fontsize=11, labelpad=10)

fig.suptitle('Diferentes Padrões e seus Espectros de Frequência',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/01_espectros_sinteticos.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observação: o padrão de grade (artefato de up-convolution) produz')
print('picos característicos no espectro FFT, ausentes em imagens naturais.')

---
## 4. Perfil Radial (Integração Azimutal)

Convertemos o espectro 2D em um perfil 1D usando a integração azimutal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cores = ['steelblue', 'tomato', 'mediumseagreen', 'darkorange']

for (nome, img), cor in zip(test_images.items(), cores):
    psd1d = compute_power_spectrum_1d(img)
    nome_limpo = nome.replace('\n', ' ')
    axes[0].plot(psd1d, label=nome_limpo, color=cor, linewidth=2)

axes[0].set_xlabel('Frequência Radial (bin)', fontsize=12)
axes[0].set_ylabel('Potência Normalizada', fontsize=12)
axes[0].set_title('Perfis Espectrais Radiais 1D', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Destaque do artefato de grade
psd_grade = compute_power_spectrum_1d(test_images['Padrão de Grade\n(Artefato Up-Conv)'])
psd_gradiente = compute_power_spectrum_1d(test_images['Gradiente Suave\n(Baixa Frequência)'])

x = np.arange(len(psd_grade))
axes[1].plot(psd_gradiente, color='steelblue', label='Natural (Gradiente)', linewidth=2)
axes[1].plot(psd_grade, color='tomato', label='GAN-like (Grade)', linewidth=2, linestyle='--')
axes[1].fill_between(x, psd_gradiente, psd_grade,
                     where=(psd_grade > psd_gradiente),
                     alpha=0.3, color='tomato', label='Excesso espectral (distorção)')
axes[1].set_xlabel('Frequência Radial (bin)', fontsize=12)
axes[1].set_ylabel('Potência Normalizada', fontsize=12)
axes[1].set_title('Distorção Espectral: Natural vs. GAN-like', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Análise de Perfis Radiais — Integração Azimutal',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/02_perfis_radiais.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Simulação do Artefato de Up-Convolution

O artefato central identificado no artigo é que operações de up-convolution (convolução transposta) criam padrões periódicos no espectro de frequência. Vamos simular isso manualmente.

In [ ]:
np.random.seed(0)
size = 128

# --- Imagem "real" natural (baixa frequência dominante) ---
def make_natural_image(size):
    """Simula uma imagem natural com frequências baixas dominantes."""
    # Mapa de ruído suavizado (Gaussiano)
    noise = np.random.randn(size, size)
    kernel_size = 15
    sigma = 5
    blurred = cv2.GaussianBlur(noise, (kernel_size, kernel_size), sigma)
    blurred = (blurred - blurred.min()) / (blurred.max() - blurred.min())
    return blurred

# --- Imagem simulando saída de GAN (com artefato de up-conv stride=2) ---
def make_gan_artifact_image(size, stride=2):
    """
    Simula o artefato espectral de up-convolution com stride.
    Up-convolution com stride=2 introduz padrões a cada 1/stride da frequência.
    """
    natural = make_natural_image(size)
    # Padrão periódico (checkerboard) introduzido pela up-conv
    checkerboard = np.indices((size, size)).sum(axis=0) % stride
    artifact = natural + 0.15 * checkerboard
    artifact = np.clip(artifact, 0, 1)
    return artifact

# Gera as imagens
n_samples = 50
imagens_reais = [make_natural_image(size) for _ in range(n_samples)]
imagens_gan = [make_gan_artifact_image(size, stride=2) for _ in range(n_samples)]

# Calcula espectros médios
n_bins = 60
spectra_real = np.stack([compute_power_spectrum_1d(img)[:n_bins] for img in imagens_reais])
spectra_gan = np.stack([compute_power_spectrum_1d(img)[:n_bins] for img in imagens_gan])

mean_real = spectra_real.mean(axis=0)
std_real = spectra_real.std(axis=0)
mean_gan = spectra_gan.mean(axis=0)
std_gan = spectra_gan.std(axis=0)

print(f'Amostras reais: {len(imagens_reais)}')
print(f'Amostras GAN:   {len(imagens_gan)}')
print(f'Bins espectrais: {n_bins}')

In [ ]:
# Visualiza exemplos de cada conjunto
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(5):
    axes[0, i].imshow(imagens_reais[i], cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Imagens Reais', fontsize=11)

    axes[1, i].imshow(imagens_gan[i], cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Imagens GAN', fontsize=11)

fig.suptitle('Amostras do Conjunto de Dados Sintético', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/03_amostras_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comparação dos espectros — réplica da Figura 1 do artigo
plot_spectrum_comparison(
    mean_real, std_real,
    mean_gan, std_gan,
    label_real='Imagens Reais (naturais)',
    label_fake='Imagens GAN (com artefato up-conv)',
    title='Espectro de Potência Azimutal: Real vs. GAN\n'
          '(Reprodução da Figura 1 — Durall et al., CVPR 2020)',
    save_path='../results/04_comparacao_espectros.png'
)

---
## 6. Análise Estatística da Separabilidade Espectral

Verificamos quantitativamente se os perfis espectrais são discriminativos o suficiente para separar imagens reais de geradas.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Prepara dataset para classificação
X = np.vstack([spectra_real, spectra_gan])     # features: perfil espectral 1D
y = np.array([0] * n_samples + [1] * n_samples)  # 0=real, 1=GAN

# Normaliza as features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Redução de dimensionalidade para visualização
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

# Classificador linear simples (como no artigo)
clf = LogisticRegression(random_state=42, max_iter=1000)
scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')

print(f'Acurácia de classificação (5-fold CV):')
print(f'  Média:  {scores.mean():.2%}')
print(f'  Desvio: {scores.std():.2%}')
print()
print('O artigo original reporta até 100% de acurácia em benchmarks públicos.')

In [ ]:
# Visualização PCA
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter PCA
scatter = axes[0].scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=y, cmap='RdBu', alpha=0.7, edgecolors='white', linewidths=0.5, s=60
)
plt.colorbar(scatter, ax=axes[0], ticks=[0, 1],
             label='0 = Real  |  1 = GAN')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)', fontsize=11)
axes[0].set_title('PCA do Espaço Espectral\n(Separabilidade Visual)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Scores de acurácia por fold
fold_names = [f'Fold {i+1}' for i in range(len(scores))]
bars = axes[1].bar(fold_names, scores * 100, color='steelblue', alpha=0.8, edgecolor='navy')
axes[1].axhline(scores.mean() * 100, color='tomato', linestyle='--',
                linewidth=2, label=f'Média: {scores.mean():.1%}')
axes[1].set_ylim(0, 110)
axes[1].set_ylabel('Acurácia (%)', fontsize=11)
axes[1].set_title('Acurácia de Classificação por Fold\n(Regressão Logística)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

for bar, score in zip(bars, scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{score:.1%}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Detectabilidade de Imagens GAN pelo Espectro de Frequência',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/05_classificacao_espectral.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Resumo dos Resultados

| Métrica | Valor |
|---------|-------|
| Acurácia média (5-fold CV) | ver saída acima |
| Método de classificação | Regressão Logística sobre perfil espectral 1D |
| Features | Perfil radial azimutal normalizado (60 bins) |

### Conclusões

1. **As distorções espectrais são detectáveis**: o perfil espectral 1D é suficiente para separar imagens reais de geradas por GAN com alta acurácia.

2. **O artefato de up-convolution**: operações de up-convolution com stride introduzem energia espectral excedente em frequências médias/altas, ausente em imagens naturais.

3. **Simplicidade do classificador**: um classificador linear simples (Regressão Logística) é suficiente, confirmando que as features espectrais são linearmente separáveis — exatamente como demonstrado em Durall et al. (2020).

---
*Próximo notebook: `02_deteccao_deepfakes.ipynb` — aplicação em imagens reais e análise comparativa de diferentes arquiteturas de GAN.*